## Stage 1 Training

In [7]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade_from_scratch \
#   --quality-loss \
#   --use-highres-refine \
#   --epochs 24 \
#   --batch-size 8 \
#   --lr 5e-5 \
#   --diagnostics-after-training \
#   --diag-save-dir runs/stage1_quality_upgrade_from_scratch/diagnostics_stage1 \
#   --diag-balanced-samples-per-class 8 \
#   --diag-max-batches 50 \
#   --diag-num-samples 12

In [8]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade \
#   --warm-start runs/stage1_quality_upgrade_from_scratch/checkpoints/stage1_best.pt \
#   --allow-partial-load \
#   --quality-loss \
#   --use-highres-refine \
#   --epochs 24 \
#   --batch-size 8 \
#   --lr 5e-5 \
#   --presence-threshold 0.25 \
#   --topk-presence-k 128 \
#   --small-part-area-tau 0.015 \
#   --small-part-weight-max 6.0 \
#   --small-part-weight-power 0.5 \
#   --quality-presence-bce 0.40 \
#   --valid-absent-topmean-fp 0.08 \
#   --valid-absent-mean-fp 0.02 \
#   --invalid-part-topmean 0.35 \
#   --invalid-part-mean 0.08 \
#   --gt-support-leak 0.35 \
#   --pred-support-containment 0.25 \
#   --boundary-loss 0.08 \
#   --focal-functional 0.12 \
#   --tversky-functional 0.12 \
#   --quality-topq 0.02 \
#   --diagnostics-after-training \
#   --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1 \
#   --diag-balanced-samples-per-class 8 \
#   --diag-max-batches 50 \
#   --diag-num-samples 12 \
#   --diag-max-parts-per-sample 8 \
#   --diag-mask-threshold 0.40

## 4. Diagnostics-only run for an existing checkpoint

In [9]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade \
#   --diag-checkpoint runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
#   --allow-partial-load \
#   --quality-loss \
#   --use-highres-refine \
#   --presence-threshold 0.25 \
#   --topk-presence-k 128 \
#   --diagnostics-only \
#   --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1 \
#   --diag-balanced-samples-per-class 8 \
#   --diag-max-batches 50 \
#   --diag-num-samples 12 \
#   --diag-max-parts-per-sample 8 \
#   --diag-mask-threshold 0.40 \
#   --diag-presence-thresholds 0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50

## 5. Fast diagnostics on a small subset

In [10]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade \
#   --diag-checkpoint runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
#   --allow-partial-load \
#   --quality-loss \
#   --use-highres-refine \
#   --presence-threshold 0.25 \
#   --topk-presence-k 128 \
#   --diagnostics-only \
#   --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1_fast \
#   --diag-balanced-samples-per-class 2 \
#   --diag-max-batches 8 \
#   --diag-num-samples 6 \
#   --diag-max-parts-per-sample 8 \
#   --diag-mask-threshold 0.40

## Build Strict AOG v9


In [11]:
!PYTHONPATH=src python scripts/calibrate_strict_aog_terminals.py \
  --config configs/stage1_quality_upgrade.yaml \
  --stage1-ckpt runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
  --out runs/strict_aog_cache_v17/terminal_calibration.json \
  --split val \
  --device auto \
  --batch-size 8

[dataset] train.json: 20457 samples | C=11 F=13 R=40
[dataset] val.json: 1205 samples | C=11 F=13 R=40
[datasets] train: ../full_hyco/PartImageNet/annotations/train/train.json | images: ../full_hyco/PartImageNet/images/train | samples: 20457
[datasets] val: ../full_hyco/PartImageNet/annotations/val/val.json | images: ../full_hyco/PartImageNet/images/val | samples: 1205
/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/dfli/anaconda3/envs/partviz/lib/python3.12/

In [12]:
!PYTHONPATH=src python scripts/cache_strict_aog_terminals.py \
  --config configs/stage1_quality_upgrade.yaml \
  --stage1-ckpt runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
  --out-dir runs/strict_aog_cache_v17 \
  --device auto \
  --splits train,val \
  --batch-size 8 \
  --num-workers 8 \
  --threshold 0.30 \
  --terminal-calibration-json runs/strict_aog_cache_v17/terminal_calibration.json \
  --support-gate-mode post \
  --min-presence 0.02 \
  --max-terminals 48 \
  --mask-size 64 \
  --support-power 1.0 \
  --min-support-overlap 0.15 \
  --support-component-mode best \
  --support-component-threshold 0.35 \
  --shard-size 1024 \
  --store-images \
  --store-images-splits val

[dataset] train.json: 20457 samples | C=11 F=13 R=40
[dataset] val.json: 1205 samples | C=11 F=13 R=40
[datasets] train: ../full_hyco/PartImageNet/annotations/train/train.json | images: ../full_hyco/PartImageNet/images/train | samples: 20457
[datasets] val: ../full_hyco/PartImageNet/annotations/val/val.json | images: ../full_hyco/PartImageNet/images/val | samples: 1205
/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/dfli/anaconda3/envs/partviz/lib/python3.12/

In [ ]:
!PYTHONPATH=src python scripts/build_strict_aog.py \
  --config configs/stage1_quality_upgrade.yaml \
  --cache runs/strict_aog_cache_v17/train_strict_aog_terminals.pt \
  --out runs/strict_aog_cache_v17/strict_aog_v17.pt \
  --num-templates-per-class 3 \
  --max-slots-per-template 10 \
  --max-slots-per-part 4 \
  --required-tau 0.45 \
  --min-slot-support 0.12 \
  --min-role-overlap 0.02 \
  --min-edge-support 0.08 \
  --min-edge-count 2 \
  --min-edge-information-gain 0.02 \
  --max-edges-per-template 18 \
  --relation-var-floor 0.006 \
  --geom-var-floor 0.004 \
  --count-max 6 \
  --count-smoothing 1.0 \
  --count-var-floor 0.25 \
  --count-support-tau 0.10 \
  --peer-jaccard-tau 0.20

saved strict AOG to runs/strict_aog_cache_v15/strict_aog_v15.pt
classes=11 templates=3 slots=10 edges=211
valid_templates=33 valid_slots=166
edge_info_gain mean=0.9244 max=3.0134
global_relation_background nonempty_part_pairs=73
part_count_constraints active_dims=115
valid_templates_without_edges=1/33


## Train Stage 2: Strict AOG v13 Count GPU-MF

In [1]:
!PYTHONPATH=src python scripts/train_strict_aog.py \
  --grammar runs/strict_aog_cache_v17/strict_aog_v17.pt \
  --train-cache runs/strict_aog_cache_v17/train_strict_aog_terminals.pt \
  --val-cache runs/strict_aog_cache_v17/val_strict_aog_terminals.pt \
  --save-dir runs/strict_aog_v18_count_llr_role_floor \
  --device cuda \
  --batch-size 128 \
  --num-workers 0 \
  --preload-cache \
  --epochs 20 \
  --lr 1e-4 \
  --assignment gpu_mf \
  --mf-iters 2 \
  --mf-tau 0.50 \
  --mf-column-iters 6 \
  --mf-edge-chunk-size 128 \
  --edge-score-mode peer_llr \
  --edge-info-gain-power 0.5 \
  --node-score-normalization none \
  --edge-score-normalization sqrt \
  --class-prior-weight 0.0 \
  --relation-weight 0.10 \
  --role-overlap-weight 0.40 \
  --role-overlap-floor 0.02 \
  --count-weight 0.20 \
  --count-model categorical \
  --count-score-mode peer_llr \
  --count-source assigned \
  --count-role-power 0.5 \
  --min-parse-role-overlap 0.20 \
  --low-role-penalty 0.75 \
  --min-parse-inst-edges 2.0 \
  --low-inst-edge-penalty 0.75 \
  --min-parse-edge-coverage 0.40 \
  --low-edge-coverage-penalty 0.75 \
  --edge-missing-weight 0.75 \
  --node-app-weight 0.30 \
  --node-geom-weight 0.35 \
  --node-presence-weight 0.05 \
  --slot-prior-weight 0.02 \
  --missing-weight 0.45 \
  --spurious-weight 0.05 \
  --edge-aux-weight 0.05 \
  --node-aux-weight 0.0 \
  --margin-weight 0.05 \
  --margin 0.50 \
  --template-tau 0.75 \
  --progress-every 10

[strict-aog][train] epoch=1 batch=10/160 loss=0.3721 acc=0.9883 edge_frac=0.124 count_frac=0.302 role=0.769 edge_cov=0.587 valid_pen=0.293 reuse=0.0000 edge_miss=2.185 inst_edges=2.706 elapsed=2.0s
[strict-aog][train] epoch=1 batch=20/160 loss=0.3759 acc=0.9832 edge_frac=0.124 count_frac=0.299 role=0.768 edge_cov=0.590 valid_pen=0.296 reuse=0.0000 edge_miss=2.168 inst_edges=2.679 elapsed=3.4s
[strict-aog][train] epoch=1 batch=30/160 loss=0.3776 acc=0.9828 edge_frac=0.125 count_frac=0.297 role=0.769 edge_cov=0.591 valid_pen=0.293 reuse=0.0000 edge_miss=2.154 inst_edges=2.688 elapsed=4.6s
[strict-aog][train] epoch=1 batch=40/160 loss=0.3783 acc=0.9826 edge_frac=0.126 count_frac=0.296 role=0.769 edge_cov=0.587 valid_pen=0.290 reuse=0.0000 edge_miss=2.185 inst_edges=2.697 elapsed=5.8s
[strict-aog][train] epoch=1 batch=50/160 loss=0.3803 acc=0.9812 edge_frac=0.126 count_frac=0.295 role=0.769 edge_cov=0.587 valid_pen=0.289 reuse=0.0000 edge_miss=2.181 inst_edges=2.692 elapsed=7.1s
[strict-ao